In [ ]:
import ast
import os
import re
import torch
import pandas as pd

DATA_DIR = globals().get("DATA_DIR", "/content/snuaichallenge_data")
TEST_CSV = globals().get("TEST_CSV", os.path.join(DATA_DIR, "test.csv"))
TEST_IMAGE_DIR = globals().get("TEST_IMAGE_DIR", os.path.join(DATA_DIR, "test"))

test_df = pd.read_csv(TEST_CSV)
test_df["Id"] = test_df["Id"].astype(str)


def parse_model_output(output_text):
    pattern = r"\[\s*[1-4]\s*,\s*[1-4]\s*,\s*[1-4]\s*,\s*[1-4]\s*\]"
    match = re.search(pattern, output_text)

    if match is None:
        return None

    try:
        result = ast.literal_eval(match.group())
    except (ValueError, SyntaxError, TypeError):
        return None

    if isinstance(result, list) and len(result) == 4 and sorted(result) == [1, 2, 3, 4]:
        return [int(value) for value in result]

    return None

# --- 추론 함수 정의 ---
def get_prompt_message(row, image_dir):
    """
    4장의 프레임과 문장을 조합하여 Qwen2-VL-2B-Instruct 모델에 보낼 프롬프트 메시지를 구성합니다.
    """
    img_files = [row['Input_1'], row['Input_2'], row['Input_3'], row['Input_4']]
    sentence = row['Sentence']

    content = []
    for i, img_file in enumerate(img_files):
        img_path = os.path.join(image_dir, str(row['Id']), str(img_file))
        content.append({"type": "text", "text": f"\nInput image {i+1}:"})
        content.append({
            "type": "image",
            "image": img_path,
        })

    user_text = (
        f"The video is described as: \"{sentence}\"\n\n"
        "The four images are shuffled frames from the video. "
        "Compare the visual states and temporal progression.\n"
        "Return the actual temporal rank of Input image 1, "
        "Input image 2, Input image 3, and Input image 4 "
        "in that order.\n"
        "Respond only with one Python-style permutation list."
    )
    content.append({"type": "text", "text": user_text})

    messages = [
        {
            "role": "user",
            "content": content,
        }
    ]
    return messages

In [ ]:
from qwen_vl_utils import process_vision_info

row = test_df.iloc[0]

messages = get_prompt_message(row, TEST_IMAGE_DIR)

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)

inputs = inputs.to(model.device)

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False
    )

generated_ids_trimmed = [
    out_ids[len(in_ids):]
    for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]

output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

print("모델 원본 출력:", output_text)
print("제출 형식 변환:", parse_model_output(output_text))

In [ ]:
import ast
import re


def parse_model_output(output_text):
    """
    ?? ???? [1, 2, 3, 4] ??? ??? ????.

    ???? ?? ??? ?? ?????:
    [Input_1? ?? ??, Input_2? ?? ??, Input_3? ?? ??, Input_4? ?? ??]
    """
    pattern = r"\[\s*[1-4]\s*,\s*[1-4]\s*,\s*[1-4]\s*,\s*[1-4]\s*\]"
    match = re.search(pattern, output_text)

    if match is None:
        return None

    try:
        result = ast.literal_eval(match.group())
    except (ValueError, SyntaxError, TypeError):
        return None

    if isinstance(result, list) and len(result) == 4 and sorted(result) == [1, 2, 3, 4]:
        return [int(value) for value in result]

    return None


In [ ]:
from tqdm.auto import tqdm
predictions = []

print("Starting inference...")

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    messages = get_prompt_message(row, TEST_IMAGE_DIR)

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    inputs = inputs.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    pred_list = parse_model_output(output_text)

    predictions.append({
        "Id": row["Id"],
        "Answer": str(pred_list if pred_list is not None else [1, 2, 3, 4])
    })

In [ ]:
submit_path = "/content/outputs/submission.csv"
os.makedirs(os.path.dirname(submit_path), exist_ok=True)

submission_df = pd.DataFrame(predictions)
submission_df.to_csv(submit_path, index=False)

print("저장 완료:", submit_path)
display(submission_df.head())